# JalanKita — YOLOv8 Training

| Kelas | Label |
|-------|-------|
| 0 | lubang_besar |
| 1 | lubang_kecil |
| 2 | retak_kulit_buaya |
| 3 | retak_memanjang |

---
## Auto-Checkpoint
Notebook ini **auto-upload checkpoint** ke Kaggle Dataset `marcillleeeee/jalankita-checkpoint`
setiap 25 epoch via `kaggle datasets version`. Jika session terputus:
1. Buka notebook **kaggle_resume** (import terpisah)
2. Set **Input dataset** ke `marcillleeeee/jalankita-checkpoint`
3. Run all — resume otomatis dari epoch terakhir

**Sebelum session habis**, pastikan download `jalankita_final_model.zip` dari sidebar kanan.

---
## 1. Setup

In [ ]:
import os, sys, subprocess, json, shutil, threading, time, glob, yaml, zipfile
import locale
locale.getpreferredencoding = lambda: 'UTF-8'

!pip install -q ultralytics

from ultralytics import YOLO
import torch
from pathlib import Path
from IPython.display import display, Image as DImage

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


---
## 2. Extract Dataset

In [ ]:
# Auto-discover dataset path — cari data.yaml di /kaggle/input/
import glob as _glob
yaml_candidates = _glob.glob('/kaggle/input/**/data.yaml', recursive=True)
if not yaml_candidates:
    raise FileNotFoundError('data.yaml tidak ditemukan di /kaggle/input/')
dataset_path = Path(yaml_candidates[0]).parent
DATASET_DIR = str(dataset_path)

print(f'Dataset path: {DATASET_DIR}')

# Copy data.yaml ke working dir dengan absolute path (YOLO butuh path absolut)
with open(dataset_path / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = DATASET_DIR
DATA_YAML = '/kaggle/working/data_fixed.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_cfg, f)
print(f'Fixed data.yaml -> {DATA_YAML}')

for split in ['train', 'valid', 'test']:
    img_dir = dataset_path / split / 'images'
    lbl_dir = dataset_path / split / 'labels'
    n_imgs = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob('*'))) if lbl_dir.exists() else 0
    print(f'{split:6s}: {n_imgs:>6} images, {n_lbls:>6} labels')
print(f'Classes: {data_cfg["names"]}')


---
## 3. Training YOLOv8n

Callback auto-upload checkpoint ke `marcillleeeee/jalankita-checkpoint` tiap 25 epoch.
Upload berjalan **synchronous** (5-15 detik), training pause sebentar.

**PERTAMA KALI JALAN?** Sebelum training, buka Kaggle Dataset baru:
1. https://www.kaggle.com/datasets -> New Dataset
2. Title: `JalanKita Checkpoint`, slug otomatis `jalankita-checkpoint`
3. Upload file dummy, Create
4. Selesai — callback akan `kaggle datasets version` ke dataset ini

Kalau lupa, callback gagal di epoch 25 tapi training tetap lanjut.
Download `last.pt` manual, upload ke dataset, lalu resume.

In [ ]:
# ── Upload callback ──────────────────────────────────
CHECKPOINT_DATASET = 'marcillleeeee/jalankita-checkpoint'
CHECKPOINT_DIR = '/kaggle/working/checkpoint_upload'
CHECKPOINT_INTERVAL = 25

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

meta_path = os.path.join(CHECKPOINT_DIR, 'dataset-metadata.json')
if not os.path.exists(meta_path):
    with open(meta_path, 'w') as f:
        json.dump({
            "title": "JalanKita Checkpoint",
            "id": CHECKPOINT_DATASET,
            "licenses": [{"name": "CC0-1.0"}]
        }, f)

def upload_checkpoint(trainer):
    epoch = trainer.epoch + 1
    if epoch % CHECKPOINT_INTERVAL != 0:
        return
    if epoch >= trainer.epochs:
        return

    weights_dir = trainer.save_dir / 'weights'
    for fname in ['last.pt', 'best.pt']:
        src = os.path.join(weights_dir, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(CHECKPOINT_DIR, fname))

    try:
        result = subprocess.run(
            ['kaggle', 'datasets', 'version', '-p', CHECKPOINT_DIR,
             '-m', f'Checkpoint epoch {epoch}', '-q'],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode != 0:
            print('  ! Version failed, trying create...')
            result = subprocess.run(
                ['kaggle', 'datasets', 'create', '-p', CHECKPOINT_DIR, '-q'],
                capture_output=True, text=True, timeout=120
            )
        if result.returncode == 0:
            print(f'  \u2713 Checkpoint epoch {epoch} uploaded')
        else:
            print(f'  \u2717 Upload failed: {result.stderr.strip()[:200]}')
    except Exception as e:
        print(f'  \u2717 Upload error: {e}')


# ── Parameter Training ──
BATCH_SIZE = 32
EPOCHS = 150
IMGSZ = 640
PATIENCE = 30

# ── Fresh Training ──
model = YOLO('yolov8n.pt')
model.add_callback('on_train_epoch_end', upload_checkpoint)

print(f'Model: YOLOv8n ({sum(p.numel() for p in model.model.parameters()):,} params)')
print(f'Batch: {BATCH_SIZE}, Epochs: {EPOCHS}, Img Size: {IMGSZ}')

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    patience=PATIENCE,
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    cos_lr=True,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=30.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.2,
    close_mosaic=15,
    save=True,
    save_period=10,
    val=True,
    amp=True,
    workers=4,
    seed=42,
    deterministic=False,
    project='jalankita_train',
    name='yolov8n_final',
    exist_ok=True,
)

print('\nTraining selesai!')

EXP_DIR = '/kaggle/working/jalankita_train/yolov8n_final'
BEST_PT = f'{EXP_DIR}/weights/best.pt'


---
## 4. Evaluasi — Test Set

In [ ]:
if os.path.exists(BEST_PT):
    best_path = BEST_PT
elif os.path.exists(f'{EXP_DIR}/weights/last.pt'):
    best_path = f'{EXP_DIR}/weights/last.pt'
else:
    raise FileNotFoundError('No model found!')

best_model = YOLO(best_path)
val_results = best_model.val(
    data=DATA_YAML,
    split='test',
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    conf=0.25,
    iou=0.5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=True
)

print('\n' + '='*60)
print('HASIL VALIDASI — TEST SET')
print('='*60)
print(f'mAP50-95:  {val_results.box.map:.4f}')
print(f'mAP50:     {val_results.box.map50:.4f}')
print(f'mAP75:     {val_results.box.map75:.4f}')
print(f'Precision: {val_results.box.mp:.4f}')
print(f'Recall:    {val_results.box.mr:.4f}')
f1 = 2 * val_results.box.mp * val_results.box.mr / (val_results.box.mp + val_results.box.mr + 1e-10)
print(f'F1-score:  {f1:.4f}')

print('\nPer-Class:')
for i, name in enumerate(best_model.names.values()):
    ap50 = val_results.box.ap50[i]
    ap5095 = val_results.box.ap[i].mean()
    p = val_results.box.p[i]
    r = val_results.box.r[i]
    f1 = 2 * p * r / (p + r + 1e-10)
    print(f'  {name:22s}: P={p:.4f} R={r:.4f} F1={f1:.4f} mAP50={ap50:.4f} mAP50-95={ap5095:.4f}')


---
## 5. Visualisasi

In [ ]:
cm = glob.glob(f'{EXP_DIR}/confusion_matrix*.png')
if not cm:
    cm = glob.glob(f'{EXP_DIR}/**/confusion_matrix*.png', recursive=True)
if cm:
    display(DImage(cm[-1]))
else:
    print('No confusion matrix found')


In [ ]:
preds = glob.glob(f'{EXP_DIR}/**/val_batch*_pred*', recursive=True)
if preds:
    display(DImage(preds[0]))
else:
    print('No validation sample found')


---
## 6. Export + Download

In [ ]:
best_model.export(format='onnx', imgsz=IMGSZ)
best_model.export(format='torchscript', imgsz=IMGSZ)

print('Exported models:')
!ls -lh {EXP_DIR}/weights/


In [ ]:
output_zip = '/kaggle/working/jalankita_final_model.zip'
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(EXP_DIR):
        for f in files:
            if f.endswith(('.pt', '.onnx', '.torchscript', '.png', '.csv', '.yaml')):
                fp = os.path.join(root, f)
                arc = os.path.relpath(fp, EXP_DIR)
                zf.write(fp, arc)

print(f'Model package: {output_zip}')
print(f'Size: {os.path.getsize(output_zip) / 1e6:.1f} MB')
print()
print('Download: sidebar kanan -> /kaggle/working/jalankita_final_model.zip')


---
## 7. Jika Session Terputus

Auto-checkpoint terupload tiap 25 epoch ke `marcillleeeee/jalankita-checkpoint`.

Lanjutkan:
1. Import **kaggle_resume.ipynb** ke Kaggle
2. Add Input dataset:
   - `marcillleeeee/jalankita-dataset-final` (gambar)
   - `marcillleeeee/jalankita-checkpoint` (checkpoint)
3. Run All

Auto-upload gagal? Download manual:
- `/kaggle/working/jalankita_train/yolov8n_final/weights/last.pt`
- Upload ke dataset via web UI, lalu resume